# 69. The Shortest Path Problem

## Tier 2: The Classic Heuristic (Python Implementation)

### Key assumptions
- Network edges have non-negative weights (required for Dijkstra's algorithm)
- Priority queue implementation ensures efficient node selection
- Graph is connected (path exists from source to all other nodes)
- Real-time performance is more important than guaranteed optimality

### Approach (step-by-step)
1. **Algorithm Selection**: Implement Dijkstra's algorithm with priority queue
2. **Data Structures**: Use distance array, priority queue, and predecessor tracking
3. **Initialization**: Set source distance to 0, all others to infinity
4. **Main Loop**: Extract minimum distance node, relax outgoing edges
5. **Path Reconstruction**: Use predecessor array to build final path
6. **Performance Analysis**: Measure computational complexity and runtime

### What to look for in the results
- Optimal path distance matching mathematical formulation
- Computational efficiency compared to exact methods
- Priority queue operations and node visitation order
- Scalability analysis for larger network instances

### Concrete example (from the source)
Freight network with 15 major distribution hubs across North America:
- Nodes: LA, Phoenix, Dallas, Chicago, NYC, etc.
- Expected optimal path: LA -> Phoenix -> Dallas -> Chicago -> NYC
- Expected distance: 2200 miles with computational efficiency suitable for real-time operations

In [1]:
# Import required libraries for Dijkstra's algorithm implementation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import heapq
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Set
import time
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Python heap queue available: {hasattr(heapq, 'heappush')}")

In [ ]:
@dataclass
class FreightNetwork:
    """Represents a freight transportation network"""
    nodes: Dict[str, Tuple[float, float]]  # node_id -> (x, y) coordinates
    edges: Dict[str, Dict[str, float]]     # adjacency list: node -> {neighbor: distance}
    node_names: Dict[str, str]             # node_id -> full name

@dataclass
class DijkstraResult:
    """Results from Dijkstra's algorithm"""
    distances: Dict[str, float]
    predecessors: Dict[str, Optional[str]]
    visited_order: List[str]
    computation_time: float
    iterations: int

print("Data structures defined")

In [ ]:
def create_freight_network() -> FreightNetwork:
    """Create the freight network from the concrete example"""
    
    # Define node coordinates for visualization
    nodes = {
        'LA': (0, 0),      # Los Angeles
        'PHX': (2, 1),     # Phoenix
        'DEN': (1, 2),     # Denver
        'DAL': (3, 1),     # Dallas
        'CHI': (5, 2),     # Chicago
        'ATL': (6, 1),     # Atlanta
        'NYC': (8, 2),     # New York
        'BOS': (9, 3),     # Boston
        'MIA': (7, -1),    # Miami
        'DET': (5.5, 2.5), # Detroit
        'HOU': (4, -1),    # Houston
        'SEA': (0, 3),     # Seattle
        'POR': (0.5, 2.5), # Portland
        'MIN': (4.5, 3),   # Minneapolis
        'KCM': (4, 2)      # Kansas City
    }
    
    # Define node full names
    node_names = {
        'LA': 'Los Angeles',
        'PHX': 'Phoenix',
        'DEN': 'Denver',
        'DAL': 'Dallas',
        'CHI': 'Chicago',
        'ATL': 'Atlanta',
        'NYC': 'New York',
        'BOS': 'Boston',
        'MIA': 'Miami',
        'DET': 'Detroit',
        'HOU': 'Houston',
        'SEA': 'Seattle',
        'POR': 'Portland',
        'MIN': 'Minneapolis',
        'KCM': 'Kansas City'
    }
    
    # Define adjacency list with distances (in miles, simplified for demonstration)
    edges = {
        'LA': {'PHX': 370, 'DEN': 1015, 'SEA': 1135},
        'PHX': {'LA': 370, 'DEN': 586, 'DAL': 887, 'KCM': 1150},
        'DEN': {'LA': 1015, 'PHX': 586, 'KCM': 600, 'MIN': 920, 'POR': 1245, 'SEA': 1305},
        'DAL': {'PHX': 887, 'KCM': 490, 'ATL': 795, 'HOU': 239, 'CHI': 925},
        'CHI': {'KCM': 515, 'MIN': 409, 'DET': 281, 'DAL': 925, 'ATL': 717},
        'ATL': {'CHI': 717, 'DAL': 795, 'MIA': 661, 'NYC': 888},
        'NYC': {'ATL': 888, 'BOS': 215, 'CHI': 790},
        'BOS': {'NYC': 215, 'DET': 713},
        'MIA': {'ATL': 661, 'HOU': 1185},
        'DET': {'CHI': 281, 'BOS': 713, 'MIN': 527, 'NYC': 614},
        'HOU': {'DAL': 239, 'MIA': 1185, 'KCM': 695},
        'SEA': {'LA': 1135, 'DEN': 1305, 'POR': 175},
        'POR': {'SEA': 175, 'DEN': 1245},
        'MIN': {'DEN': 920, 'CHI': 409, 'DET': 527, 'KCM': 398},
        'KCM': {'DEN': 600, 'PHX': 1150, 'DAL': 490, 'CHI': 515, 'HOU': 695, 'MIN': 398}
    }
    
    return FreightNetwork(nodes, edges, node_names)

# Create the freight network
network = create_freight_network()
print(f"Freight network created with {len(network.nodes)} nodes")
print(f"Network nodes: {list(network.nodes.keys())}")
print(f"Total edges: {sum(len(neighbors) for neighbors in network.edges.values())}")

In [ ]:
def dijkstra_shortest_path(network: FreightNetwork, source: str, target: str) -> DijkstraResult:
    """Implement Dijkstra's algorithm with priority queue"""
    
    start_time = time.time()
    
    # Initialize distances to infinity, except source
    distances = {node: float('inf') for node in network.nodes}
    distances[source] = 0
    
    # Initialize predecessors
    predecessors = {node: None for node in network.nodes}
    
    # Priority queue: (distance, node)
    priority_queue = [(0, source)]
    
    # Track visited nodes and visitation order
    visited = set()
    visited_order = []
    iterations = 0
    
    while priority_queue:
        iterations += 1
        
        # Extract node with minimum distance
        current_distance, current_node = heapq.heappop(priority_queue)
        
        # Skip if we've already found a better path
        if current_node in visited:
            continue
            
        visited.add(current_node)
        visited_order.append(current_node)
        
        # Early termination if we reached the target
        if current_node == target:
            break
        
        # Relax all outgoing edges
        for neighbor, edge_weight in network.edges.get(current_node, {}).items():
            if neighbor not in visited:
                new_distance = current_distance + edge_weight
                
                # If we found a better path, update and add to queue
                if new_distance < distances[neighbor]:
                    distances[neighbor] = new_distance
                    predecessors[neighbor] = current_node
                    heapq.heappush(priority_queue, (new_distance, neighbor))
    
    computation_time = time.time() - start_time
    
    return DijkstraResult(distances, predecessors, visited_order, computation_time, iterations)

def reconstruct_path(predecessors: Dict[str, Optional[str]], target: str) -> List[str]:
    """Reconstruct path from source to target using predecessors"""
    path = []
    current = target
    
    while current is not None:
        path.append(current)
        current = predecessors[current]
    
    return path[::-1]  # Reverse to get source-to-target order

print("Dijkstra algorithm functions defined")

In [ ]:
# Solve shortest path from LA to NYC
source = 'LA'
target = 'NYC'

result = dijkstra_shortest_path(network, source, target)
optimal_path = reconstruct_path(result.predecessors, target)

print("=== DIJKSTRA'S ALGORITHM RESULTS ===")
print(f"Source: {source} ({network.node_names[source]})")
print(f"Target: {target} ({network.node_names[target]})")
print(f"Shortest distance: {result.distances[target]:.0f} miles")
print(f"Optimal path: {' -> '.join(optimal_path)}")
print(f"Computation time: {result.computation_time*1000:.3f} milliseconds")
print(f"Iterations: {result.iterations}")
print(f"Nodes visited: {len(result.visited_order)}")

print("\nPath details:")
total_distance = 0
for i in range(len(optimal_path) - 1):
    from_node = optimal_path[i]
    to_node = optimal_path[i + 1]
    segment_distance = network.edges[from_node][to_node]
    total_distance += segment_distance
    print(f"  {from_node} -> {to_node}: {segment_distance:.0f} miles")

print(f"\nTotal distance: {total_distance:.0f} miles")
print(f"Verification: {total_distance:.0f} == {result.distances[target]:.0f} ✓")

In [ ]:
def visualize_dijkstra_solution(network: FreightNetwork, result: DijkstraResult, 
                                source: str, target: str, optimal_path: List[str]):
    """Visualize the Dijkstra algorithm solution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Network graph with optimal path
    ax1.set_title('Freight Network - Optimal Path', fontsize=14, fontweight='bold')
    
    # Draw all edges (gray)
    for from_node, neighbors in network.edges.items():
        for to_node, distance in neighbors.items():
            from_coords = network.nodes[from_node]
            to_coords = network.nodes[to_node]
            
            # Check if edge is in optimal path
            is_optimal = (from_node in optimal_path and to_node in optimal_path and 
                         optimal_path.index(from_node) + 1 == optimal_path.index(to_node))
            
            color = 'red' if is_optimal else 'gray'
            linewidth = 3 if is_optimal else 1
            alpha = 0.8 if is_optimal else 0.3
            
            ax1.annotate('', xy=to_coords, xytext=from_coords,
                       arrowprops=dict(arrowstyle='->', color=color, lw=linewidth, alpha=alpha))
    
    # Draw nodes
    for node_id, coords in network.nodes.items():
        color = 'red' if node_id == source else 'blue' if node_id == target else 'lightgreen'
        size = 400 if node_id in optimal_path else 300
        ax1.scatter(coords[0], coords[1], s=size, c=color, edgecolors='black', linewidth=2, zorder=5)
        ax1.text(coords[0], coords[1]-0.2, node_id, fontsize=10, ha='center', fontweight='bold')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # Plot 2: Node visitation order
    ax2.set_title('Node Visitation Order', fontsize=14, fontweight='bold')
    visitation_data = [(node, result.distances[node]) for node in result.visited_order]
    nodes_ordered, distances_ordered = zip(*visitation_data)
    
    colors = ['red' if node == target else 'blue' if node == source else 'green' for node in nodes_ordered]
    bars = ax2.bar(range(len(nodes_ordered)), distances_ordered, color=colors, alpha=0.7)
    ax2.set_xticks(range(len(nodes_ordered)))
    ax2.set_xticklabels(nodes_ordered, rotation=45)
    ax2.set_ylabel('Distance from Source (miles)')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, (bar, distance) in enumerate(zip(bars, distances_ordered)):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 50,
                f'{distance:.0f}', ha='center', va='bottom', fontsize=8)
    
    # Plot 3: Distance progression
    ax3.set_title('Distance Progression During Algorithm', fontsize=14, fontweight='bold')
    
    # Show how distances improve over iterations
    distance_progression = []
    temp_distances = {node: float('inf') for node in network.nodes}
    temp_distances[source] = 0
    
    for i, visited_node in enumerate(result.visited_order):
        current_distances = temp_distances.copy()
        distance_progression.append((i, visited_node, current_distances[target]))
        
        # Simulate distance update (simplified)
        if visited_node in network.edges and target in network.edges[visited_node]:
            potential_distance = temp_distances[visited_node] + network.edges[visited_node][target]
            if potential_distance < temp_distances[target]:
                temp_distances[target] = potential_distance
    
    iterations, nodes_in_progress, target_distances = zip(*distance_progression)
    ax3.plot(iterations, target_distances, 'b-o', linewidth=2, markersize=6)
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('Distance to Target (miles)')
    ax3.grid(True, alpha=0.3)
    
    # Mark final distance
    ax3.axhline(y=result.distances[target], color='red', linestyle='--', alpha=0.7, 
                label=f'Final: {result.distances[target]:.0f} miles')
    ax3.legend()
    
    # Plot 4: Algorithm performance metrics
    ax4.set_title('Performance Metrics', fontsize=14, fontweight='bold')
    
    metrics = ['Nodes\nVisited', 'Iterations', 'Time\n(ms)', 'Path\nLength']
    values = [len(result.visited_order), result.iterations, 
              result.computation_time * 1000, len(optimal_path)]
    
    colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
    bars = ax4.bar(metrics, values, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
    
    # Add value labels
    for bar, value in zip(bars, values):
        height = bar.get_height()
        if metrics[2] == 'Time\n(ms)':  # Time in milliseconds
            label = f'{value:.3f}'
        else:
            label = f'{int(value)}'
        ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.05,
                label, ha='center', va='bottom', fontweight='bold')
    
    ax4.set_ylabel('Value')
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

# Visualize the solution
visualize_dijkstra_solution(network, result, source, target, optimal_path)

In [ ]:
def find_k_shortest_paths(network: FreightNetwork, source: str, target: str, k: int = 3) -> List[Tuple[List[str], float]]:
    """Find k shortest paths using a modified approach"""
    
    # For demonstration, we'll find alternative paths by temporarily removing edges
    
    # First, find the primary shortest path
    primary_result = dijkstra_shortest_path(network, source, target)
    primary_path = reconstruct_path(primary_result.predecessors, target)
    primary_distance = primary_result.distances[target]
    
    paths = [(primary_path, primary_distance)]
    
    # Find alternative paths by removing edges from the primary path
    for i in range(len(primary_path) - 1):
        if len(paths) >= k:
            break
            
        # Create a modified network by removing one edge from the primary path
        from_node = primary_path[i]
        to_node = primary_path[i + 1]
        
        modified_edges = {}
        for node, neighbors in network.edges.items():
            modified_edges[node] = neighbors.copy()
            if node == from_node and to_node in modified_edges[node]:
                del modified_edges[node][to_node]
        
        # Create modified network
        modified_network = FreightNetwork(network.nodes, modified_edges, network.node_names)
        
        # Find shortest path in modified network
        try:
            alt_result = dijkstra_shortest_path(modified_network, source, target)
            if alt_result.distances[target] < float('inf'):
                alt_path = reconstruct_path(alt_result.predecessors, target)
                alt_distance = alt_result.distances[target]
                
                # Check if this path is different from existing ones
                is_duplicate = False
                for existing_path, _ in paths:
                    if alt_path == existing_path:
                        is_duplicate = True
                        break
                
                if not is_duplicate:
                    paths.append((alt_path, alt_distance))
        except:
            continue
    
    return paths[:k]

# Find top 3 shortest paths
k_paths = find_k_shortest_paths(network, source, target, k=3)

print("=== TOP 3 SHORTEST PATHS ===")
for i, (path, distance) in enumerate(k_paths, 1):
    print(f"{i}. Distance: {distance:.0f}, Route: {' -> '.join(path)}")

print("\nAlternative path analysis:")
if len(k_paths) > 1:
    primary_distance = k_paths[0][1]
    for i in range(1, len(k_paths)):
        alt_distance = k_paths[i][1]
        percentage_increase = ((alt_distance - primary_distance) / primary_distance) * 100
        print(f"  Alternative {i}: +{percentage_increase:.1f}% distance increase")
else:
    print("  No alternative paths found")

In [ ]:
def scalability_analysis():
    """Analyze algorithm scalability with different network sizes"""
    
    print("=== SCALABILITY ANALYSIS ===")
    
    # Test with different subsets of the network
    test_cases = [
        {'nodes': ['LA', 'PHX', 'DEN', 'CHI', 'NYC'], 'name': 'Small (5 nodes)'},
        {'nodes': ['LA', 'PHX', 'DEN', 'DAL', 'CHI', 'ATL', 'NYC'], 'name': 'Medium (7 nodes)'},
        {'nodes': ['LA', 'PHX', 'DEN', 'DAL', 'CHI', 'ATL', 'NYC', 'BOS', 'MIA'], 'name': 'Large (9 nodes)'},
        {'nodes': list(network.nodes.keys()), 'name': 'Full (15 nodes)'}
    ]
    
    results = []
    
    for test_case in test_cases:
        test_nodes = test_case['nodes']
        test_name = test_case['name']
        
        # Create sub-network
        sub_edges = {}
        for node in test_nodes:
            sub_edges[node] = {}
            for neighbor, distance in network.edges.get(node, {}).items():
                if neighbor in test_nodes:
                    sub_edges[node][neighbor] = distance
        
        sub_nodes = {node: network.nodes[node] for node in test_nodes}
        sub_names = {node: network.node_names[node] for node in test_nodes}
        
        sub_network = FreightNetwork(sub_nodes, sub_edges, sub_names)
        
        # Run Dijkstra's algorithm
        if 'LA' in test_nodes and 'NYC' in test_nodes:
            start_time = time.time()
            sub_result = dijkstra_shortest_path(sub_network, 'LA', 'NYC')
            end_time = time.time()
            
            results.append({
                'name': test_name,
                'nodes': len(test_nodes),
                'edges': sum(len(neighbors) for neighbors in sub_edges.values()),
                'time_ms': (end_time - start_time) * 1000,
                'iterations': sub_result.iterations,
                'distance': sub_result.distances['NYC']
            })
    
    # Display results
    print(f"{'Network Size':<15} {'Nodes':<8} {'Edges':<8} {'Time (ms)':<12} {'Iterations':<12} {'Distance':<10}")
    print("-" * 75)
    
    for result in results:
        print(f"{result['name']:<15} {result['nodes']:<8} {result['edges']:<8} "
              f"{result['time_ms']:<12.3f} {result['iterations']:<12} {result['distance']:<10.0f}")
    
    # Complexity analysis
    print("\n=== TIME COMPLEXITY ANALYSIS ===")
    print("Theoretical complexity: O((V + E) log V)")
    print("Where V = number of vertices, E = number of edges")
    
    # Empirical analysis
    if len(results) >= 2:
        small_time = results[0]['time_ms']
        large_time = results[-1]['time_ms']
        small_nodes = results[0]['nodes']
        large_nodes = results[-1]['nodes']
        
        node_ratio = large_nodes / small_nodes
        time_ratio = large_time / small_time if small_time > 0 else float('inf')
        
        print(f"\nEmpirical scaling:")
        print(f"  Network size increased by factor: {node_ratio:.1f}x")
        print(f"  Computation time increased by factor: {time_ratio:.1f}x")
        print(f"  Expected (logarithmic) scaling factor: ~{np.log(node_ratio):.1f}x")
    
    return results

# Perform scalability analysis
scalability_results = scalability_analysis()

In [ ]:
def compare_with_mathematical_formulation():
    """Compare Dijkstra's algorithm with mathematical formulation approach"""
    
    print("=== COMPARISON WITH MATHEMATICAL FORMULATION ===")
    
    # Create a smaller network for exact comparison
    comparison_network = FreightNetwork(
        nodes={'A': (0, 0), 'B': (1, 1), 'C': (2, 0), 'D': (3, 1), 'E': (4, 0)},
        edges={
            'A': {'B': 3, 'C': 2},
            'B': {'C': 1, 'D': 4},
            'C': {'B': 1, 'D': 6, 'E': 2},
            'D': {'E': 4},
            'E': {}
        },
        node_names={'A': 'A', 'B': 'B', 'C': 'C', 'D': 'D', 'E': 'E'}
    )
    
    # Solve with Dijkstra
    dijkstra_result = dijkstra_shortest_path(comparison_network, 'A', 'E')
    dijkstra_path = reconstruct_path(dijkstra_result.predecessors, 'E')
    
    print("Dijkstra's Algorithm:")
    print(f"  Path: {' -> '.join(dijkstra_path)}")
    print(f"  Distance: {dijkstra_result.distances['E']}")
    print(f"  Time: {dijkstra_result.computation_time*1000:.3f} ms")
    
    # Manual calculation for verification
    print("\nManual Verification:")
    paths_and_distances = [
        (['A', 'B', 'C', 'E'], 3 + 1 + 2),  # A->B->C->E
        (['A', 'C', 'E'], 2 + 2),          # A->C->E
        (['A', 'B', 'D', 'E'], 3 + 4 + 4),  # A->B->D->E
        (['A', 'C', 'B', 'D', 'E'], 2 + 1 + 4 + 4),  # A->C->B->D->E
        (['A', 'C', 'D', 'E'], 2 + 6 + 4),  # A->C->D->E
    ]
    
    best_path, best_distance = min(paths_and_distances, key=lambda x: x[1])
    
    print(f"  Best path: {' -> '.join(best_path)}")
    print(f"  Best distance: {best_distance}")
    
    # Comparison
    distance_match = abs(dijkstra_result.distances['E'] - best_distance) < 0.001
    path_match = dijkstra_path == best_path
    
    print(f"\nVerification:")
    print(f"  Distance matches: {distance_match} ✓" if distance_match else f"  Distance matches: {distance_match} ✗")
    print(f"  Path matches: {path_match} ✓" if path_match else f"  Path matches: {path_match} ✗")
    
    if distance_match and path_match:
        print("\n🎉 Perfect match! Dijkstra's algorithm found the optimal solution.")
    
    # Performance comparison
    print("\n=== PERFORMANCE COMPARISON ===")
    print("Dijkstra's Algorithm Advantages:")
    print("  ✓ Computationally efficient for large networks")
    print("  ✓ Guaranteed optimal for non-negative edge weights")
    print("  ✓ Simple implementation with priority queue")
    print("  ✓ Early termination when target is reached")
    
    print("\nMathematical Formulation Advantages:")
    print("  ✓ Provides optimality proof")
    print("  ✓ Handles additional constraints naturally")
    print("  ✓ Suitable for multi-objective optimization")
    print("  ✓ Foundation for advanced optimization techniques")

# Compare with mathematical formulation
compare_with_mathematical_formulation()

### Why this Tier exists vs Tier 1
Tier 2 provides practical advantages over the mathematical formulation:
- **Computational Efficiency**: O((V + E) log V) complexity vs exponential for MIP
- **Real-time Performance**: Millisecond execution suitable for dynamic routing
- **Scalability**: Handles large networks efficiently without optimization software
- **Implementation Simplicity**: Straightforward algorithm without complex solvers
- **Early Termination**: Stops when target is reached, doesn't need to explore entire network

### Pros / Cons vs Tier 1 (Mathematical Formulation)
**Pros:**
- Much faster execution time (milliseconds vs seconds/minutes)
- Scales to large networks with thousands of nodes
- No external dependencies or optimization software required
- Guaranteed optimal for non-negative edge weights
- Simple to implement and understand
- Early termination capability

**Cons:**
- Limited to non-negative edge weights
- Cannot handle additional constraints (capacity, flow, etc.)
- Single objective optimization only
- No optimality proof or sensitivity analysis built-in
- Less flexible for complex problem variations
- Requires separate algorithm for each source-target pair

### When to use this Tier
- **Real-time routing applications** requiring fast responses
- **Large-scale networks** with thousands of nodes and edges
- **Navigation systems** and GPS applications
- **Network routing protocols** and telecommunications
- **Logistics and supply chain** daily operations
- **Web services** and APIs requiring quick responses

### Key takeaways
Dijkstra's algorithm provides the practical foundation for shortest path optimization:
- Priority queue implementation ensures efficient node selection
- Greedy approach with optimal substructure property
- Guaranteed optimality for non-negative edge weights
- Excellent scalability for large real-world networks
- Foundation for many advanced routing algorithms and protocols